# Task 4

## Task: Banknote Authentication Classification

In this task, you will work with the dataset provided in the file **`data_banknote_authentication.csv`**

Your objective is to build and evaluate classification models to predict the authenticity of banknotes.

You should:

- Load and explore the dataset.
- Build two classification models:
  - **Decision Tree**
  - **Random Forest**
- Use **GridSearchCV** to optimize the hyperparameters of each model.
- Evaluate the performance of both models using:
  - **Confusion Matrix**
  - **Classification Report**

Finally, **compare the performance of the two models** and discuss which model performs better for this dataset.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [4]:
df = pd.read_csv("data_banknote_authentication.csv")

# 3. Explore dataset
print("First 5 rows:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nColumn names:")
print(df.columns)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
# Target distribution
df["Class"].value_counts()

# Optional: statistical summary
print("\nStatistical Summary:")
print(df.describe())

First 5 rows:
   Variance_Wavelet  Skewness_Wavelet  Curtosis_Wavelet  Image_Entropy  Class
0           3.62160            8.6661           -2.8073       -0.44699      0
1           4.54590            8.1674           -2.4586       -1.46210      0
2           3.86600           -2.6383            1.9242        0.10645      0
3           3.45660            9.5228           -4.0112       -3.59440      0
4           0.32924           -4.4552            4.5718       -0.98880      0

Dataset shape: (1372, 5)

Column names:
Index(['Variance_Wavelet', 'Skewness_Wavelet', 'Curtosis_Wavelet',
       'Image_Entropy', 'Class'],
      dtype='object')

Missing values:
Variance_Wavelet    0
Skewness_Wavelet    0
Curtosis_Wavelet    0
Image_Entropy       0
Class               0
dtype: int64

Data types:
Variance_Wavelet    float64
Skewness_Wavelet    float64
Curtosis_Wavelet    float64
Image_Entropy       float64
Class                 int64
dtype: object

Statistical Summary:
       Variance_Wavelet  

In [6]:
X = df.drop("Class", axis=1)
y = df["Class"]
X.head()
y.head()

,Class
0,0
1,0
2,0
3,0
4,0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((1097, 4), (275, 4))

In [8]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    estimator=dt_model,
    param_grid=dt_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 3, 5, 10, 15, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [9]:
dt_grid.best_params_
dt_grid.best_score_

np.float64(0.9936197592361976)

In [12]:
best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)

accuracy_score(y_test, y_pred_dt)
confusion_matrix(y_test, y_pred_dt)
cm_dt = confusion_matrix(y_test, y_pred_dt)
print("Decision Tree Confusion Matrix:")
print(cm_dt)
print(classification_report(y_test, y_pred_dt))

Decision Tree Confusion Matrix:
[[151   2]
 [  0 122]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       153
           1       0.98      1.00      0.99       122

    accuracy                           0.99       275
   macro avg       0.99      0.99      0.99       275
weighted avg       0.99      0.99      0.99       275



In [11]:
rf_model = RandomForestClassifier(random_state=42)

rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [13]:
rf_grid.best_params_
rf_grid.best_score_

np.float64(0.9945288501452885)

In [14]:
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

accuracy_score(y_test, y_pred_rf)
confusion_matrix(y_test, y_pred_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Random Forest Confusion Matrix:")
print(cm_rf)
print(classification_report(y_test, y_pred_rf))

Random Forest Confusion Matrix:
[[152   1]
 [  0 122]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       153
           1       0.99      1.00      1.00       122

    accuracy                           1.00       275
   macro avg       1.00      1.00      1.00       275
weighted avg       1.00      1.00      1.00       275



In [15]:
dt_accuracy = accuracy_score(y_test, y_pred_dt)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

dt_accuracy, rf_accuracy
if rf_accuracy > dt_accuracy:
    print("Random Forest performs better.")
elif dt_accuracy > rf_accuracy:
    print("Decision Tree performs better.")
else:
    print("Both models perform equally.")

Random Forest performs better.
